# The Mandelbrot set and Julia sets

*maths · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/maths/mandelbrot_julia)

Iterate $z \mapsto z^2 + c$ on a whole grid of complex pixels at once. Points whose orbit stays bounded form the Mandelbrot set $M$ (vary $c$, start at $z_0 = 0$) or a filled Julia set $K_c$ (fix $c$, start at the pixel). A smooth escape count and a distance estimate from the derivative make the picture sharp all the way down a zoom into the seahorse valley, while the Julia set changes shape as $c$ walks around the main cardioid.

**Vector form**

$$z_{n+1} = z_n^2 + c,\qquad z'_{n+1} = 2\,z_n z'_n + c',\qquad M = \{c : |z_n| \not\to \infty,\ z_0 = 0\},\qquad d(c) \approx \frac{|z_n|\ln|z_n|}{|z'_n|}$$

**Index (tensor) form**

$$z_{p}\leftarrow z_{p}\,z_{p} + c_{p},\qquad z'_{p}\leftarrow 2\,z_{p}\,z'_{p} + c',\qquad \nu_p = n_p + 1 - \log_2\ln|z_{p}|,\qquad d_p = \frac{|z_p|\ln|z_p|}{|z'_p|}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $c_{p}$ | `pixel_grid(centre, width, n, dev)` | (n, n) complex128 | one complex number per pixel $p$; no index is summed, every pixel runs on its own |
| $z_p \leftarrow z_p z_p + c_p$ | `z * z + c` | (P,) | one step of the quadratic map for all pixels still iterating |
| $z'_p \leftarrow 2 z_p z'_p + c'$ | `2 * z * dz + dc` | (P,) | chain rule: $c' = 1$ for $M$ (derivative in $c$), $c' = 0$ and $z'_0 = 1$ for Julia sets (derivative in $z_0$) |
| $n_p$ | `n_esc[alive[out]] = n + 1` | (P,) | first step with $/z_p/ > R$; escaped pixels are dropped from the tensors, so the work shrinks |
| $\nu_p$ | `n + 1 - torch.log2(torch.log(z_abs))` | (P,) | smooth (continuous) escape count: no colour steps between bands |
| $d_p$ | `z_abs * torch.log(z_abs) / dz_abs` | (P,) | distance estimate: pixels with $d_p$ under a quarter pixel are drawn dark, which reveals the thin filaments |
| $q(q + x - \tfrac14) \le \tfrac{y^2}{4}$ | `in_cardioid_or_bulb(c)` | (n, n) bool | exact test for the main cardioid (and $/c+1/ \le \frac14$ for the period-2 bulb), with $q = (x-\frac14)^2 + y^2$ |

Why escape at $|z| > 2$: if $|z| > \max(2, |c|)$ then $|z^2 + c| \ge |z|^2 - |c| > |z|$, and the orbit grows without bound. So every $c$ with $|c| > 2$ escapes at the first step, and $M$ lies inside the disc $|c| \le 2$. A larger bailout $R = 1000$ is used here because it makes the smooth count $\nu = n + 1 - \log_2 \ln|z_n|$ accurate: near infinity $z \mapsto z^2$ doubles $\ln|z|$ each step, and $\log_2$ undoes exactly that. The main cardioid is the set of $c$ where $z^* = z^{*2} + c$ is an attracting fixed point, $|2z^*| < 1$; writing $2z^* = w$ gives $c = w/2 - w^2/4$ with $|w| \le 1$, and the period-2 bulb is where the 2-cycle is attracting, $|4(1+c)| < 1$. These two pieces cover about 1.37 of the total area $\approx 1.5066$ of $M$, and no pixel inside them ever escapes. The distance estimate $d = |z_n|\ln|z_n|/|z'_n|$ comes from the potential $G(c) = \lim 2^{-n}\ln|z_n|$, as $G/|\nabla G|$; near the set the true distance lies within roughly a factor 2 of it (a consequence of the Koebe quarter theorem). The Julia panel keeps $c$ on the cardioid's edge, where the fixed point is neutral: $K_c$ is always connected there because $c \in M$, but its shape changes completely from one angle to the next. Each Julia set looks like $M$ near that $c$, which is why zooms into $M$ show little Julia sets. Zooms deeper than about $10^{-13}$ need more than the 16 digits of complex128.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/maths/mandelbrot_julia/main.py
"""Main loop: parameters → tensors → solve → plot."""

import math
from pathlib import Path

import torch

from problems._common.plotting import frame_indices
from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import cardioid_boundary, in_cardioid_or_bulb

M_AREA = 1.5066  # area of M, from published pixel-counting estimates (≈ 1.50659)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n = 64 if fast_mode() else params["resolution"]
    base = params["max_iter"]

    # 1. the whole set: exact cardioid/bulb tests against the iteration
    centre, width = tensors.START
    c = tensors.pixel_grid(centre, width, 2 * n, dev)  # (2n, 2n)
    whole = solve.mandelbrot(c, base)
    bounded = whole["n"] == 0
    exact = in_cardioid_or_bulb(c)
    pixel_area = (width / (2 * n - 1)) ** 2
    print(f"overview: {c.numel()} pixels, {base} iterations, bailout |z| > 1000")
    print(f"pixels inside the cardioid or the period-2 bulb that escaped: {int((exact & ~bounded).sum())} (must be 0)")
    print(
        f"cardioid + bulb area 3π/8 + π/16 = {3 * math.pi / 8 + math.pi / 16:.4f}, "
        f"pixel count {exact.sum() * pixel_area:.4f}"
    )
    print(f"area of M from the iteration ≈ {bounded.sum() * pixel_area:.4f}   (literature ≈ {M_AREA})")
    d = whole["d"][~bounded]
    print(f"distance estimate outside: min {d.min():.2e}, all positive: {bool((d > 0).all())}")

    # 2. the zoom: one tensor per frame, more iterations as the frames get deeper
    n_frames = params["frames"]
    picked = frame_indices(n_frames, n_frames)  # fast mode keeps only 5 frames
    widths_all = tensors.zoom_widths(width, params["zoom_decades"], n_frames)
    widths = [widths_all[k] for k in picked]
    centres = tensors.zoom_centres(tensors.TARGETS[params["target"]], widths)
    zoom = []
    for w, ctr in zip(widths, centres):
        zoom.append(solve.mandelbrot(tensors.pixel_grid(ctr, w, n, dev), tensors.iterations_for(w, base)))
    last = zoom[-1]
    print(
        f"zoom into {params['target']}: width {widths[0]:.1f} → {widths[-1]:.2e} (×{widths[0] / widths[-1]:.1e}), "
        f"{tensors.iterations_for(widths[-1], base)} iterations in the deepest frame, "
        f"{(last['n'] == 0).float().mean() * 100:.1f}% of its pixels still bounded"
    )

    # 3. Julia sets with c on the main cardioid's edge, all frames as one (F, n, n) batch
    theta_all = tensors.julia_angles(n_frames, dev)
    c_julia = cardioid_boundary(theta_all[picked])  # (F,)
    z0 = tensors.pixel_grid(0j, 3.2, n, dev)  # (n, n)
    jul = solve.julia(z0[None], c_julia[:, None, None], 300)  # (F, n, n)
    julia = [{k: v[f] for k, v in jul.items()} for f in range(len(picked))]
    filled = (jul["n"] == 0).float().mean(dim=(1, 2))
    for f in (0, len(picked) // 2, len(picked) - 1):
        cf = c_julia[f]
        print(
            f"Julia c = {cf.real:+.4f} {cf.imag:+.4f}i: |2z*| = {abs(1 - torch.sqrt(1 - 4 * cf)):.4f}, "
            f"filled {filled[f]:.1%}"
        )

    box = (centre.real - width / 2, centre.real + width / 2, -width / 2, width / 2)
    return plot.animate(zoom, widths, centres, julia, c_julia, whole, box, out_dir / "mandelbrot_julia.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/maths/mandelbrot_julia/equation.py
r"""Mathematics → tensor translation.

Every pixel p of the picture is a complex number. One step of the quadratic map, for all pixels at once, is

    z_p ← z_p^2 + c_p,        z'_p ← 2 z_p z'_p + c'_p.

Mandelbrot set M:  c_p = pixel, z_0 = 0, the derivative is taken with respect to c, so c' = 1 and z'_0 = 0.
Julia set J_c:     c fixed,     z_0 = pixel, the derivative is taken with respect to z_0, so c' = 0 and z'_0 = 1.

No index is summed: the pixel index p only runs along, which is why the whole picture is one tensor operation.
A point escapes once |z_n| > R; if |z| ever exceeds 2 (and |c| ≤ 2) it is certain to run off to infinity.

Smooth colouring:   ν = n + 1 − log_2(ln|z_n|)          (the integer count n made continuous)
Distance estimate:  d = |z_n| ln|z_n| / |z'_n|           (≈ distance from the pixel to the set's boundary)
"""

import torch


def step(z: torch.Tensor, dz: torch.Tensor, c: torch.Tensor, dc: float) -> tuple[torch.Tensor, torch.Tensor]:
    r"""z ← z² + c and z' ← 2 z z' + c' for every pixel.   All complex tensors of the same shape (P,)."""
    return z * z + c, 2 * z * dz + dc


def smooth_count(n: torch.Tensor, z_abs: torch.Tensor) -> torch.Tensor:
    r"""ν = n + 1 − log₂(ln|z_n|): continuous escape count, so colour bands do not show steps.   (P,) → (P,)"""
    return n + 1 - torch.log2(torch.log(z_abs))


def distance_estimate(z_abs: torch.Tensor, dz_abs: torch.Tensor) -> torch.Tensor:
    r"""d = |z| ln|z| / |z'|: distance from an escaping point to the boundary, within about a factor 2.   (P,)"""
    return z_abs * torch.log(z_abs) / dz_abs


def in_cardioid_or_bulb(c: torch.Tensor) -> torch.Tensor:
    r"""Exact tests for the two biggest pieces of M (no iteration needed).

    Main cardioid: c = w/2 − w²/4 with |w| ≤ 1, where z* = w/2 is an attracting fixed point (|2 z*| ≤ 1).
    In coordinates, q = (x − ¼)² + y² and q (q + x − ¼) ≤ y²/4.
    Period-2 bulb: z → z² + c has an attracting 2-cycle iff |4(1 + c)| ≤ 1, the disc |c + 1| ≤ ¼.
    """
    x, y = c.real, c.imag
    q = (x - 0.25) ** 2 + y**2
    cardioid = q * (q + x - 0.25) <= 0.25 * y**2
    bulb = (x + 1) ** 2 + y**2 <= 1 / 16
    return cardioid | bulb


def cardioid_boundary(theta: torch.Tensor) -> torch.Tensor:
    r"""c(θ) = e^{iθ}/2 − e^{2iθ}/4: the edge of the main cardioid, where the fixed point has |multiplier| = 1."""
    w = torch.exp(1j * theta)
    return w / 2 - w * w / 4


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/maths/mandelbrot_julia/tensors.py
"""Build the concrete tensors: complex pixel grids, the zoom path and the Julia parameters on the cardioid."""

import math

import torch

# Zoom targets on the boundary of M (complex128 needs a width above ~1e-13 before pixels merge).
TARGETS = {
    "seahorse": complex(-0.743643887037151, 0.131825904205330),  # deep in the seahorse valley
    "dendrite": complex(0.0, 1.0),  # c = i: a Misiurewicz point, 0 → i → −1+i → −i → −1+i → … (preperiodic)
    "minibrot": complex(-1.754877666246693, 0.0),  # centre of the period-3 minibrot, root of c³ + 2c² + c + 1 = 0
}
START = (complex(-0.6, 0.0), 3.2)  # centre and width of the first frame: the whole set


def pixel_grid(centre: complex, width: float, n: int, device: torch.device) -> torch.Tensor:
    """Square grid of complex numbers c_{ij} = centre + (x_j + i y_i), shape (n, n), complex128; row 0 is the top."""
    s = torch.linspace(-width / 2, width / 2, n, dtype=torch.float64, device=device)
    return centre + s[None, :] + 1j * s.flip(0)[:, None]


def zoom_widths(width0: float, decades: float, n_frames: int) -> list[float]:
    """Frame widths shrinking geometrically from width0 to width0·10^(−decades)."""
    return [width0 * 10 ** (-decades * k / max(1, n_frames - 1)) for k in range(n_frames)]


def zoom_centres(target: complex, widths: list[float]) -> list[complex]:
    """Glide from the overview centre to the target while zooming (the target stays in every frame)."""
    c0, w0 = START
    out = []
    for w in widths:
        t = min(1.0, math.log(w0 / w) / math.log(10))  # the first decade pans, then the centre is the target
        out.append(c0 + (target - c0) * (3 * t * t - 2 * t * t * t))
    return out


def iterations_for(width: float, base: int) -> int:
    """Deeper frames need more iterations: base + 150 per decade of zoom."""
    return int(base + 150 * max(0.0, math.log10(START[1] / width)))


def julia_angles(n_frames: int, device: torch.device) -> torch.Tensor:
    """Angles θ_f around the main cardioid for the Julia panel: (F,). θ = π is c = −¾, θ = 2π/3 the period-3 root."""
    return torch.linspace(0.15, 2 * math.pi - 0.15, n_frames, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/maths/mandelbrot_julia/solve.py
"""Iterate the quadratic map on every pixel at once and record how (and how fast) each one escapes."""

import torch

from problems._common.progress import steps

from .equation import distance_estimate, smooth_count, step


def escape(z0: torch.Tensor, c: torch.Tensor, dz0: float, dc: float, max_iter: int, radius: float = 1e3) -> dict:
    """Escape-time iteration on a batch of pixels of any shape.

    Mandelbrot: z0 = 0, c = pixels, dz0 = 0, dc = 1.   Julia: z0 = pixels, c = constant, dz0 = 1, dc = 0.
    Only pixels that are still bounded are updated: escaped ones are written out and dropped from the tensors.
    Returns n (escape step, 0 = never escaped), smooth count ν and distance estimate d, all with the input's shape.
    """
    shape = torch.broadcast_shapes(z0.shape, c.shape)
    c = c.expand(shape).flatten()  # (P,)
    z = z0.expand(shape).flatten().clone()
    dz = torch.full_like(z, dz0)
    alive = torch.arange(z.numel(), device=z.device)  # indices of the pixels still iterating
    n_esc = torch.zeros(z.numel(), dtype=torch.float64, device=z.device)
    z_abs = torch.ones_like(n_esc)
    dz_abs = torch.ones_like(n_esc)
    for n in steps(max_iter, "iterations"):
        z, dz = step(z, dz, c, dc)
        out = z.abs() > radius
        if out.any():
            done = alive[out]
            n_esc[done] = n + 1
            z_abs[done] = z[out].abs()
            dz_abs[done] = dz[out].abs()
            keep = ~out
            alive, z, dz, c = alive[keep], z[keep], dz[keep], c[keep]
            if alive.numel() == 0:
                break
    escaped = n_esc > 0
    nu = torch.where(escaped, smooth_count(n_esc, z_abs), torch.nan)
    dist = torch.where(escaped, distance_estimate(z_abs, dz_abs), 0.0)
    return {"n": n_esc.reshape(shape), "nu": nu.reshape(shape), "d": dist.reshape(shape)}


def mandelbrot(c: torch.Tensor, max_iter: int, radius: float = 1e3) -> dict:
    """M: start every orbit at z = 0 with the pixel as c; derivative with respect to c."""
    return escape(torch.zeros_like(c), c, 0.0, 1.0, max_iter, radius)


def julia(z0: torch.Tensor, c: torch.Tensor, max_iter: int, radius: float = 1e3) -> dict:
    """Filled Julia set K_c: start at the pixel with c fixed (c may be (F, 1, 1) for one c per frame)."""
    return escape(z0, c, 1.0, 0.0, max_iter, radius)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/maths/mandelbrot_julia/plot.py
"""Animate the zoom into M next to a Julia set whose c walks around the main cardioid, plus an overview map."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LinearSegmentedColormap

from problems._common.plotting import save_gif

# white far away → blue near the set; the set itself and its thin filaments are dark (the "bad" colour)
COLOURS = ["#ffffff", "#bfdbfe", "#60a5fa", "#2563eb", "#1e3a8a"]
CMAP = LinearSegmentedColormap.from_list("escape", COLOURS, N=16).with_extremes(bad="#111827")


def shade(result: dict, pixel: float) -> np.ma.MaskedArray:
    """log ν scaled to [0, 1] per frame; interior pixels and those within ¼ pixel of the boundary are masked."""
    nu, d = result["nu"].cpu().numpy(), result["d"].cpu().numpy()
    value = np.log(np.where(np.isnan(nu), 1.0, np.maximum(nu, 1.0)))
    lo, hi = np.nanpercentile(np.where(np.isnan(nu), np.nan, value), [1, 99])
    value = np.clip((value - lo) / max(hi - lo, 1e-9), 0, 1)
    return np.ma.masked_where(np.isnan(nu) | (d < 0.25 * pixel), value)


def animate(
    zoom: list[dict],
    widths: list[float],
    centres: list[complex],
    julia: list[dict],
    c_julia: torch.Tensor,
    overview: dict,
    overview_box: tuple[float, float, float, float],
    path: Path,
) -> Path:
    """zoom/julia: one result dict per frame with (n, n) tensors; overview: a result dict for the whole set."""
    n = zoom[0]["nu"].shape[-1]
    fig, (ax_z, ax_j, ax_o) = plt.subplots(1, 3, figsize=(10.5, 3.9))
    for ax in (ax_z, ax_j, ax_o):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
    img_z = ax_z.imshow(shade(zoom[0], widths[0] / n), cmap=CMAP, vmin=0, vmax=1, interpolation="nearest")
    img_j = ax_j.imshow(shade(julia[0], 3.2 / n), cmap=CMAP, vmin=0, vmax=1, interpolation="nearest")
    title_z = ax_z.set_title("", fontsize=9)
    title_j = ax_j.set_title("", fontsize=9)

    x0, x1, y0, y1 = overview_box
    ax_o.imshow(shade(overview, (x1 - x0) / overview["nu"].shape[-1]), cmap=CMAP, extent=overview_box, vmin=0, vmax=1)
    theta = np.linspace(0, 2 * np.pi, 400)
    w = np.exp(1j * theta)
    edge = w / 2 - w * w / 4
    ax_o.plot(edge.real, edge.imag, color="#f59e0b", lw=1.2, label="cardioid $|2z^*| = 1$")
    ax_o.plot(-1 + 0.25 * np.cos(theta), 0.25 * np.sin(theta), color="#16a34a", lw=1.2, label="bulb $|c+1| = 1/4$")
    (dot_c,) = ax_o.plot([], [], "o", color="#dc2626", ms=6, label="Julia $c$")
    box = plt.Rectangle((0, 0), 0, 0, fill=False, ec="#dc2626", lw=1.2, ls="--")
    ax_o.add_patch(box)
    ax_o.set_xlim(x0, x1)
    ax_o.set_ylim(y0, y1)
    ax_o.set_title("overview: exact tests vs iteration", fontsize=9)
    ax_o.legend(loc="lower left", fontsize=7, framealpha=0.9)

    c_np = c_julia.cpu().numpy()

    def update(f: int):
        img_z.set_data(shade(zoom[f], widths[f] / n))
        img_j.set_data(shade(julia[f], 3.2 / n))
        c, w_f = centres[f], widths[f]
        title_z.set_text(f"zoom ×{3.2 / w_f:.1e} at {c.real:+.6f} {c.imag:+.6f}i")
        title_j.set_text(f"Julia set, c = {c_np[f].real:+.3f} {c_np[f].imag:+.3f}i")
        dot_c.set_data([c_np[f].real], [c_np[f].imag])
        size = max(w_f, 0.04)  # keep the box visible once it is smaller than a pixel of the overview
        box.set_bounds(c.real - size / 2, c.imag - size / 2, size, size)

    fig.tight_layout()
    return save_gif(fig, update, list(range(len(zoom))), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'target': 'seahorse',
    'zoom_decades': 5.0,
    'resolution': 160,
    'max_iter': 300,
    'frames': 50
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.maths.mandelbrot_julia")]:
    del sys.modules[name]
main = importlib.import_module("problems.maths.mandelbrot_julia.main")
problem_dir = Path("problems/maths/mandelbrot_julia")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
